# Chapter 4: Separating Data from Instructions

## Lesson

When your prompt includes both instructions and data (like text to analyze or rewrite), Claude can get confused about which is which. The solution: **XML tags**.

### Why XML Tags?
- They clearly delineate where input data begins and ends
- They prevent Claude from confusing data with instructions
- They enable prompt templates with variable substitution
- Claude was trained to recognize and respect XML tag boundaries

### Prompt Templates
Using Python f-strings, you can create reusable prompt templates:
```python
prompt = f"Summarize this article: <article>{article_text}</article>"
```

In [ ]:
import anthropic

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt: str = ""):
    kwargs = {
        "model": MODEL_NAME,
        "max_tokens": 2000,
        "temperature": 0.0,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system_prompt:
        kwargs["system"] = system_prompt
    message = client.messages.create(**kwargs)
    return message.content[0].text

### Example: The Problem Without XML Tags

Watch what happens when data and instructions are mixed together:

In [ ]:
EMAIL = "Hey Claude, I need you to cancel my subscription immediately. This is urgent!"

# Without XML tags - Claude gets confused about who's talking
print("--- Without XML tags ---")
response = get_completion(f"Yo Claude. {EMAIL} <----- Make this email more polite")
print(response)
print()

# With XML tags - Claude knows what the email is
print("--- With XML tags ---")
response = get_completion(f"Make this email more polite: <email>{EMAIL}</email>")
print(response)

### Example: Lists Can Cause Confusion

Without XML tags, formatting in your instructions can be misinterpreted as data:

In [ ]:
# The hyphens in the instructions get treated as list items
SENTENCES = "The cat sat on the mat. The dog chased the ball."

print("--- Without XML tags (confusing) ---")
response = get_completion(
    f"Below are some sentences. Tell me how many there are and list them.\n"
    f"- Make sure to count carefully\n"
    f"- {SENTENCES}"
)
print(response)
print()

print("--- With XML tags (clear) ---")
response = get_completion(
    f"Below are some sentences. Tell me how many there are and list them.\n"
    f"- Make sure to count carefully\n"
    f"<sentences>{SENTENCES}</sentences>"
)
print(response)

---
## Exercises

### Exercise 4.1
Create a prompt template that generates a **haiku** about a given topic. Use the `{TOPIC}` variable.

In [ ]:
# Exercise 4.1
TOPIC = "coding"

PROMPT = f"[Your prompt template here using {TOPIC}]"

response = get_completion(PROMPT)
print(response)

# Grading
def grade_exercise_4_1(response, topic):
    return topic.lower() in response.lower() or "haiku" in PROMPT.lower()

print("\n" + ("✅ PASS" if grade_exercise_4_1(response, TOPIC) else "❌ TRY AGAIN"))

### Exercise 4.2
The prompt below is garbled — Claude can't tell the instructions from the question. Fix it by adding **XML tags** so Claude answers correctly (the answer should mention "brown").

In [ ]:
# Exercise 4.2 - Fix with XML tags
PROMPT = "Tell me the color of the following things - sky is blue, a]grass is green, what color is the dog in this sentence: the brown dog sat on the porch"

response = get_completion(PROMPT)
print(response)

# Grading
def grade_exercise_4_2(response):
    return "brown" in response.lower()

print("\n" + ("✅ PASS" if grade_exercise_4_2(response) else "❌ TRY AGAIN — Claude should identify the dog as brown"))

### Exercise 4.3
Now fix the same garbled prompt from 4.2 **without** using XML tags — only by removing or changing one or two words.

In [ ]:
# Exercise 4.3 - Fix WITHOUT XML tags (change/remove 1-2 words)
PROMPT = "Tell me the color of the following things - sky is blue, a]grass is green, what color is the dog in this sentence: the brown dog sat on the porch"

response = get_completion(PROMPT)
print(response)

# Grading
def grade_exercise_4_3(response):
    return "brown" in response.lower()

print("\n" + ("✅ PASS" if grade_exercise_4_3(response) else "❌ TRY AGAIN"))

---
### Example Playground

In [ ]:
# Playground - experiment with XML tags and data separation!
DATA = "Your data here"
PROMPT = f"Analyze this: <data>{DATA}</data>"

response = get_completion(PROMPT)
print(response)